<a href="https://colab.research.google.com/github/okletsdothis-lang/Roblox-useful-scripts-and-instructions/blob/main/Group%20Leaver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roblox Batch Group Leaver Script

This script is designed to leave *all* your Roblox groups and communities. **Before running, please follow the crucial step below to insert your Roblox security cookie.**

⚠️ **Important:** Roblox may still block some actions depending on current API permissions. This is the correct method, but not every account supports bulk leaving via API. This script will attempt to leave all groups found for your account.

### &#128272; **CRUCIAL STEP: Add your Roblox Security Cookie (KEEP PRIVATE!)**

**You MUST replace `"your_roblosecurity_token_here"` in the code cell below with your actual `.ROBLOSECURITY` cookie value.** This cookie is necessary for authentication. Never share this cookie with anyone as it grants access to your Roblox account.

#### **Option 1: Desktop (Chrome, Edge, Firefox)**
1. Go to `roblox.com` and log in.
2. Open Developer Tools (`F12` or `Ctrl+Shift+I`).
3. Go to the **Application** (Chrome/Edge) or **Storage** (Firefox) tab.
4. Under **Cookies**, select `https://www.roblox.com`.
5. Find the cookie named `.ROBLOSECURITY` and copy the long string in the 'Value' field.

#### **Option 2: iOS (iPhone/iPad)**
1. Download the **Orion Browser** from the App Store.
2. Install the **Cookie-editor** extension within Orion.
3. Log into `roblox.com` in Orion.
4. Open the Cookie-editor extension, find `.ROBLOSECURITY`, and copy its value.

### **What this script does:**

This script automates the process of leaving Roblox groups associated with your account using your `.ROBLOSECURITY` token.

*   **Authentication:** It uses your token to log in securely via the Roblox API.
*   **Automated Leaving:** it fetches a list of all your groups and attempts to leave them one by one.
*   **Safety Check:** It automatically **skips groups where you are the Owner** (Rank 255), as you must transfer ownership or delete the group manually before leaving.
*   **Rate Limiting:** It includes built-in delays and retry logic to avoid being blocked by Roblox for making too many requests at once.
*   **CSRF Protection:** It handles security tokens (X-CSRF) automatically to ensure requests are authorized.

In [ ]:
import requests
import time

# --- CONFIGURATION ---
ROBLOSECURITY = "your_roblosecurity_token_here"

def get_csrf(session):
    try:
        r = session.post("https://auth.roblox.com/v2/logout")
        return r.headers.get("x-csrf-token")
    except Exception as e:
        print(f"Error getting X-CSRF-TOKEN: {e}")
        return None

def run_group_leaving():
    if not ROBLOSECURITY or "your_roblosecurity" in ROBLOSECURITY:
        print("Please set a valid ROBLOSECURITY token.")
        return

    session = requests.Session()
    session.cookies.set(".ROBLOSECURITY", ROBLOSECURITY, domain=".roblox.com")
    session.headers.update({"User-Agent": "Mozilla/5.0"})

    try:
        user_res = session.get("https://users.roblox.com/v1/users/authenticated")
        user_res.raise_for_status()
        user_id = user_res.json()['id']
        print(f"Authenticated as: {user_id}")
    except:
        print("Authentication failed. Check your token.")
        return

    print("Fetching groups...")
    groups_res = session.get(f"https://groups.roblox.com/v1/users/{user_id}/groups/roles")
    groups = groups_res.json().get('data', [])

    if not groups:
        print("No groups left to leave!")
        return

    # Get token once outside the loop
    current_token = get_csrf(session)
    if not current_token:
        print("Failed to get initial CSRF token.")
        return
    session.headers["X-CSRF-TOKEN"] = current_token

    for g in groups:
        group_id = g['group']['id']
        name = g['group']['name']

        if g['role']['rank'] >= 255:
            print(f"Skipping {name} (Owner)")
            continue

        while True:
            res = session.delete(f"https://groups.roblox.com/v1/groups/{group_id}/users/{user_id}")

            if res.status_code == 200:
                print(f"[SUCCESS] Left: {name}")
                break
            elif res.status_code == 403: # Token likely expired
                print(f"Refreshing token for {name}...")
                current_token = get_csrf(session)
                session.headers["X-CSRF-TOKEN"] = current_token
                continue
            elif res.status_code == 429:
                print("Rate limit hit, waiting 5s...")
                time.sleep(5)
                continue
            else:
                print(f"Error {res.status_code} on {name}: {res.text}")
                break

        time.sleep(0.7) # Small delay to respect API limits

    print("\n[COMPLETE] Successfully finished leaving all groups!")

run_group_leaving()